In [ ]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'  # MPS fallback for ops not yet implemented on MPS (e.g. linalg_cholesky)

# 02 — Analyze

Loads training artifacts and produces paper-equivalent figures: PPO vs REINFORCE reward trajectories, paired statistical test, top-100 molecule summary statistics.

**Auto-discovers** two data sources and plots whichever is present:

- `ICML_2025_Workshop_Submission_Artifacts/pickles/` — camera-ready pickles (12 files, always present).
- `runs/` — runs produced by `notebooks/01_train.ipynb` or `scripts/run_all.py`.

Nothing in this notebook requires GPU. The `compare_paired` cells decode molecules through HierVAE and score with MGraphDTA, which need their checkpoints loaded; those cells are guarded so a recipient without GPU/MPS can still see the trajectory and Tanimoto analyses.

When the new 8-run sweep completes, re-run this notebook and the same cells will pick up the new runs alongside the camera-ready pickles for direct comparison.

## Setup

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from lisardd.analyze import plot_ppo_vs_reinforce
from lisardd.io import load_legacy_pickle, load_run

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## Discover available data

Walk `ICML_2025_Workshop_Submission_Artifacts/pickles/` and `runs/`, build a registry keyed by `(algo, reward, target)`.

In [ ]:
ARTIFACTS_DIR = repo_root / "ICML_2025_Workshop_Submission_Artifacts" / "pickles"
RUNS_DIR = repo_root / "runs"

LEGACY_FILES = {
    ("ppo", "binding", "jnk3"):     "ppo_binding_jkn3_normalize.pkl",  # typo preserved in original filename
    ("reinforce", "binding", "jnk3"): "reinforce_binding_jnk3_normalize.pkl",
    ("ppo", "binding", "egyrase"):    "ppo_binding_egyrase_normalize.pkl",
    ("reinforce", "binding", "egyrase"): "reinforce_binding_egyrase_normalize.pkl",
    ("ppo", "multi", "jnk3"):       "ppo_multi_jnk3_normalize.pkl",
    ("reinforce", "multi", "jnk3"): "reinforce_multi_jnk3_normalize.pkl",
    ("ppo", "multi", "egyrase"):    "ppo_multi_egyrase_normalize.pkl",
    ("reinforce", "multi", "egyrase"): "reinforce_multi_egyrase_normalize.pkl",
    ("ppo", "qed", "none"):         "ppo_qed.pkl",
    ("reinforce", "qed", "none"):   "reinforce_qed.pkl",
    ("ppo", "sa", "none"):          "ppo_sa.pkl",
    ("reinforce", "sa", "none"):    "reinforce_sa.pkl",
}

def _legacy_history(pkl):
    if "avg_obj_scores_ppo" in pkl:
        return {"average_obj_scores": pkl["avg_obj_scores_ppo"], "loss_actor_list": pkl.get("loss_actor_list_ppo", [])}
    return {"average_obj_scores": pkl["avg_obj_scores"], "loss_actor_list": pkl.get("loss_actor_list", [])}

legacy_runs = {}
for key, fname in LEGACY_FILES.items():
    p = ARTIFACTS_DIR / fname
    if p.exists():
        try:
            legacy_runs[key] = {
                "source": "camera_ready",
                "path": p,
                "data": load_legacy_pickle(p),
            }
        except Exception as e:
            print(f"[skip] {fname}: {e}")

new_runs = {}
if RUNS_DIR.exists():
    for run_dir in RUNS_DIR.iterdir():
        if not run_dir.is_dir() or not (run_dir / "config.yaml").exists():
            continue
        try:
            art = load_run(run_dir, load_state=False)
        except Exception as e:
            print(f"[skip] {run_dir.name}: {e}")
            continue
        cfg = art.config
        # Map reward names to legacy short labels for cross-source comparison
        reward_short = {
            "binding_affinity": "binding",
            "prop_high_binders": "binding",
            "prop_high_binders_diff": "binding",
            "multi_obj": "multi",
            "qed": "qed",
            "sa": "sa",
        }.get(cfg["reward"], cfg["reward"])
        key = (cfg["algo"], reward_short, cfg["target"])
        new_runs[key] = {
            "source": "cleaned_pipeline",
            "path": run_dir,
            "art": art,
            "config": cfg,
        }

print(f"Camera-ready runs found: {len(legacy_runs)}")
print(f"Cleaned-pipeline runs found: {len(new_runs)}")

## Reward trajectories — PPO vs REINFORCE

One panel per `(target, reward)` config. Camera-ready trajectories are plotted alongside cleaned-pipeline trajectories where both are available. Note: y-axis units differ between sources for the `binding` configs because the camera-ready binding runs used raw pKd while the cleaned pipeline uses `prop_high_binders_diff` ([0, 1] sigmoid composite). See `ICML_2025_Workshop_Submission_Artifacts/README.md` for the full delta.

In [ ]:
configs = [
    ("binding", "jnk3",   "JNK3 — binding affinity"),
    ("binding", "egyrase","gyrA — binding affinity"),
    ("multi",   "jnk3",   "JNK3 — multi-objective"),
    ("multi",   "egyrase","gyrA — multi-objective"),
    ("qed",     "none",   "QED only"),
    ("sa",      "none",   "SA only"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (reward, target, title) in enumerate(configs):
    ax = axes[idx]
    plotted_any = False

    for source_label, source_dict, color_ppo, color_rein in [
        ("camera-ready", legacy_runs, "orange",     "steelblue"),
        ("cleaned",      new_runs,    "darkorange", "navy"),
    ]:
        ppo_key = ("ppo", reward, target)
        rein_key = ("reinforce", reward, target)

        if ppo_key in source_dict:
            d = source_dict[ppo_key]
            hist = _legacy_history(d["data"]) if d["source"] == "camera_ready" else d["art"].history
            ax.plot(hist["average_obj_scores"], color=color_ppo, alpha=0.85,
                    label=f"PPO ({source_label})")
            plotted_any = True

        if rein_key in source_dict:
            d = source_dict[rein_key]
            hist = _legacy_history(d["data"]) if d["source"] == "camera_ready" else d["art"].history
            ax.plot(hist["average_obj_scores"], color=color_rein, alpha=0.85,
                    label=f"REINFORCE ({source_label})")
            plotted_any = True

    ax.set_title(title)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Average reward")
    ax.grid(True, alpha=0.3)
    if plotted_any:
        ax.legend(loc="best", fontsize=8)
    else:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes, color="gray")

plt.tight_layout()
plt.show()

## Top-100 molecule summary statistics

Reproduces paper Table 1: mean QED, SA, and within-batch Tanimoto similarity for the 100 highest-reward molecules of each run. pKd computation against the scoring model is heavy and skipped here; if needed, run the scorer separately and merge.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, Descriptors, QED
import os, sys
_SA = os.path.join(os.environ.get('RDBASE', ''), 'Contrib', 'SA_Score')
if _SA and _SA not in sys.path:
    sys.path.append(_SA)
try:
    import sascorer
    has_sa = True
except Exception:
    has_sa = False
    print("sascorer not on path; SA stats will be NaN")

def _qed(s):
    m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
    return QED.qed(m) if m is not None else np.nan

def _sa(s):
    if not has_sa: return np.nan
    m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
    return sascorer.calculateScore(m) if m is not None else np.nan

def _mean_tanimoto(smiles):
    fps = []
    for s in smiles:
        m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
        if m is not None:
            fps.append(AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048))
    if len(fps) < 2:
        return np.nan
    sims = []
    for i in range(len(fps)):
        for j in range(i + 1, len(fps)):
            sims.append(DataStructs.TanimotoSimilarity(fps[i], fps[j]))
    return float(np.mean(sims))

def _top100_smiles(record):
    if record["source"] == "camera_ready":
        return record["data"].get("best_smiles", []) or []
    df = record["art"].top100
    return df["smiles"].dropna().tolist()

rows = []
for key, record in {**legacy_runs, **new_runs}.items():
    algo, reward, target = key
    smiles = _top100_smiles(record)
    smiles = [s for s in smiles if isinstance(s, str)][:100]
    if len(smiles) == 0:
        continue
    rows.append({
        "source": record["source"],
        "algo": algo,
        "reward": reward,
        "target": target,
        "n_valid": len(smiles),
        "qed_mean": float(np.nanmean([_qed(s) for s in smiles])),
        "sa_mean": float(np.nanmean([_sa(s) for s in smiles])),
        "tanimoto_mean": _mean_tanimoto(smiles),
    })

summary = pd.DataFrame(rows).sort_values(["target", "reward", "algo", "source"]).reset_index(drop=True)
summary

## Top molecule grids

Visualize the highest-reward molecules from each available run. One grid per run, capped at 16 molecules per grid.

In [ ]:
from rdkit.Chem import Draw

def _show_grid(record, n=16):
    smiles = _top100_smiles(record)
    smiles = [s for s in smiles if isinstance(s, str)][:n]
    if not smiles:
        return None
    mols = [Chem.MolFromSmiles(s) for s in smiles]
    legends = [f"{i+1}" for i in range(len(mols))]
    return Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(220, 220), legends=legends, useSVG=False)

# Show a couple of representative grids inline; iterate the full set if you want all of them
representative_keys = [
    ("reinforce", "multi", "jnk3"),
    ("ppo", "multi", "jnk3"),
]

for key in representative_keys:
    if key in legacy_runs:
        algo, reward, target = key
        print(f"--- {algo} | {reward} | {target} | camera-ready ---")
        img = _show_grid(legacy_runs[key])
        if img is not None: display(img)
    if key in new_runs:
        algo, reward, target = key
        print(f"--- {algo} | {reward} | {target} | cleaned-pipeline ---")
        img = _show_grid(new_runs[key])
        if img is not None: display(img)

## Paired t-test: PPO vs REINFORCE

Matches the camera-ready Figure 3 statistical analysis. For each `(reward, target)` config where both PPO and REINFORCE actors are available, sample a shared batch of latents from N(0, I), transform them with each actor's policy, decode + score the resulting molecules, and run a paired t-test on per-trial mean rewards.

**Heavy:** loads HierVAE + MGraphDTA. Skip this section if you don't have the checkpoints handy.

In [ ]:
RUN_T_TEST = False  # flip to True after the new 8-run sweep completes, or to recompute against camera-ready pickles

if RUN_T_TEST:
    from lisardd.agents.networks import Actor, ActorReinforce
    from lisardd.analyze import compare_paired
    from lisardd.generators import HierVAEGenerator
    from lisardd.scoring import MGraphDTAScorer
    from lisardd.targets import get_legacy_sequence, get_sequence
    from lisardd.rewards import reward_binding_affinity, reward_multi_obj

    generator = HierVAEGenerator(
        vocab_path=repo_root / "data" / "chembl" / "recovered_vocab_2000.txt",
        ckpt_path=repo_root / "vae_model" / "vae_model.ckpt",
        device=device,
    )

    test_configs = [
        ("binding", "jnk3"),
        ("binding", "egyrase"),
        ("multi",   "jnk3"),
        ("multi",   "egyrase"),
    ]

    rows = []
    for reward, target in test_configs:
        ppo_key = ("ppo", reward, target)
        rein_key = ("reinforce", reward, target)

        # Default to camera-ready pickles for this comparison; switch to new_runs as needed.
        if ppo_key not in legacy_runs or rein_key not in legacy_runs:
            continue

        # Camera-ready pickles were trained against legacy sequences; use the matching sequence.
        if target == "none":
            target_seq = None
        else:
            target_seq = get_legacy_sequence(target)

        scorer = MGraphDTAScorer(
            target_protein=target_seq or "M",
            ckpt_path=repo_root / "score_model_weights" / "best_scoring_model.pt",
            device=device,
        )
        reward_fn = (reward_binding_affinity(scorer)
                     if reward == "binding" else
                     reward_multi_obj(scorer))

        result = compare_paired(
            ppo_actor_state=legacy_runs[ppo_key]["data"]["actor_state_dict"],
            reinforce_actor_state=legacy_runs[rein_key]["data"]["actor_state_dict"],
            decoder=generator.decoder,
            reward_fn=reward_fn,
            n_samples=100,
            n_trials=10,
            device=device,
        )
        rows.append({"reward": reward, "target": target, **{k: v for k, v in result.items() if not isinstance(v, list)}})

    pd.DataFrame(rows)
else:
    print("Skipped. Set RUN_T_TEST = True at the top of this cell to run the paired comparison.")

## Done

Next follow-ups:

- **8-run sweep** of the cleaned pipeline (overnight on Colab T4) to populate `runs/` with the new methodologically-aligned trajectories. Re-run this notebook afterward.
- **Stage 6 Vina pipeline** to validate the top-100 molecules from each run with site-targeted AutoDock Vina docking, using PDB 3FI2 (JNK3) and an E. coli gyrA holo for the box derivation. See `notebooks/03_vina_validation.ipynb` (in progress).